# ISID223 -  Introducción a los Sistemas de Información
______________________________________
- Fecha: 04-13-2026
- Prof. Iván Carrera EPN-FIS
- Nombre: Jordy Cartagena
- GR1CD
______________________________________

## S1: Introducción a Python para Sistemas de Información - Transformando Datos en Información
En este ejercicio, se pondrá en práctica lo aprendido en la _Guía01_, mediante el uso de un cuaderno de Jupyter, el uso de Pandas y análisis descriptivo estadístico básico. 
- Se cargará la librería Pandas para iniciar con el análisis exploratorio de datos.

In [1]:
import pandas as pd

### Lectura de Datos:
Una vez cargada las librerías, se procede a cargar el archivo _titanic.csv_ como un dataframe.

In [2]:
# Se puede usar df.info() para visualizar los datos. 
df = pd.read_csv('titanic.csv', sep=',')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


### Limpieza de Datos
Como se puede observar, existen varios tipos de datos no identificados (Object), los cuales pueden ser convertidos a un tipo de datos más óptimo.

In [3]:
df['Name'] = df['Name'].astype('string')
df['Sex'] = df['Sex'].astype('string')
df['Ticket'] = df['Ticket'].astype('string')
df['Cabin'] = df['Cabin'].astype('string')
df['Embarked'] = df['Embarked'].astype('string')
#Como 'Survived' está catalogado como una variable del tipo int, es necesario comprender que esto no es más que una variable cualitativa binaia.
df['Survived'] = df['Survived'].astype('string')
#De igual manera, para la variable 'Pclass' existen categorias del 1 al 3 el cual representan la clase de cada pasajero. 
df['Pclass'] = df['Pclass'].astype('string')
#Volvemos a visualizar los datos para confirmar si se han cambiado su tipo de dato.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    string 
 2   Pclass       891 non-null    string 
 3   Name         891 non-null    string 
 4   Sex          891 non-null    string 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    string 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    string 
 11  Embarked     889 non-null    string 
dtypes: float64(2), int64(3), string(7)
memory usage: 83.7 KB


Cada dato ahora tiene un tipo de dato legible. Sin embargo, es necesario filtrar datos irrelevantes que pueden causar cesgos o simplemente no aportan en nada al proximo analisis. 

In [4]:
#Usando drop, se puede remover datos como el ID de los pasajeros, y sus nombres los cuales son irrelevantes para la descripcion.
df = df.drop(columns=['PassengerId', 'Name'])
#Una vez eliminadas las columnas, para una mejor legibilidad de los datos, se proceden a crear dataframes por tipos de datos
df_num = df.select_dtypes(exclude=['string'])
df_str = df.select_dtypes(exclude=['int64', 'float'])

### Análisis Descriptivo: 
Una vez casteado los datos, es necesario conocer un breve resumen estadistico de los datos. Para ello, se usará otra funcion de la librería pandas.

In [5]:
df_num.describe()

,Age,SibSp,Parch,Fare
count,714.000000,891.000000,891.000000,891.000000
mean,29.699118,0.523008,0.381594,32.204208
std,14.526497,1.102743,0.806057,49.693429
min,0.420000,0.000000,0.000000,0.000000
25%,20.125000,0.000000,0.000000,7.910400
50%,28.000000,0.000000,0.000000,14.454200
75%,38.000000,1.000000,0.000000,31.000000
max,80.000000,8.000000,6.000000,512.329200


In [6]:
df_str.describe()

,Survived,Pclass,Sex,Ticket,Cabin,Embarked
count,891,891,891,891,204,889
unique,2,3,2,681,147,3
top,0,3,male,1601,G6,S
freq,549,491,577,7,4,644


De los _dataframes_, se puede concluir brevemente que: 
- El tamaño de la poblacion es de: $N = 891$.
- Al menos 577 de ellos son hombres.
- El promedio de edad de los pasajeros es de aproximadamente 30 años (29.69 años para ser exactos).
- Al menos 549 pasajeros perdieron la vida (segun el dataset). 


### Generación de Información: 
Una vez se puede visualizar los estadisticos descriptivos, se puede empezar con los análisis requeridos: 

#### _Análisis de supervivencia_: Calcula el porcentaje de supervivencia de los pasajeros
Para calcular el procentaje de supervivencia de los pasajeros, se usará la siguiente fórmula: 
$$[1] S_{p} = \dfrac{S}{N} \times 100 $$
Donde $S_{p}$ es el porcentaje de supervivientes, $S$ es la cantidad de supervivientes y $N$ el tamaño de la población.
Para ello, se puede observar que el valor con mayor frecuencia es '0', el cual indica la cantidad de pasajeros que no sobrevivieron, al cual denotaremos como $S'$ (el complemento del conjunto de pasajeros sobrevivientes). 
Es fácil entender que el valor de $S'$ es igual a: 
$$[2] S' = N - S$$
Por lo que reajustando [2] en [1], obtenemos: 
$$[3] S_{p} = \dfrac{N - S'}{N} \times 100$$

Una vez reajustado, podemos saber que el **porcentaje de supervivencia de los pasajeros** fue de: 

$$ S_{p} = \dfrac{891 - 549}{891} \times 100 = 38,38\%$$


#### _Análisis de clase_: Calcula el porcentaje de pasajeros según cada clase (Parch).
Para calcular el porcentaje de pasajeros según cada clase, es necesario visualizar las categorías que existen dentro de la variable cualitativa. Para ello, es necesario crear otra función usando pandas.

In [7]:
#Usando value_counts, se puede observar cada categoria dentro de la variable y la cantidad de cada una. 
df['Pclass'].value_counts()

Pclass
3    491
1    216
2    184
Name: count, dtype: Int64

Ahora, para calcular los porcentajes utilizaremos la siguiente fórmula: 
$$ P_{c\%} = \dfrac{P_{c}}{N} \times 100 $$
Donde $P_{c}$ es la cantidad de pasajeros por clase y $N$ es el tamaño de la población. De esta manera, creando una función en **Python** se obtiene que:

In [9]:
parch = [491, 216, 184]

for i, j in enumerate(parch): 
    Pc = (j/891)*100
    Pc = round(Pc, 2)
    print(f"Para la categoría {i} el porcentaje de pasajeros es del: {Pc}%.")

Para la categoría 0 el porcentaje de pasajeros es del: 55.11%.
Para la categoría 1 el porcentaje de pasajeros es del: 24.24%.
Para la categoría 2 el porcentaje de pasajeros es del: 20.65%.


#### _Análisis por Género_: Determina cuántos hombres y mujeres viajaron en el Titanic.
Para determinar la cantidad de hombres y mujeres que viajaron en el Titanic, podemos usar el mismo principio del complemento de un conjunto. Siendo $H$ (hombres) el complemento de $M$ (mujeres). El cual, se obtiene que: 
$$N = H + M$$
En el análisis exploratorio, se observa que el valor de $H$ es de 577, pues _male_ es el valor con mayor frecuencia dentro de la variable *Sex*. Es así que se puede obtener que el valor de $M$ es de: 
$$M = N - H = 314$$
Por lo que se concluye que: 
- Hubieron 577 pasajeros varones y 314 damas abordo del Titanic.

Para comprobarlo, se puede usar la misma funcion para observar la cantidad de cada categoria existente dentro de la variable, usando Pandas. 

In [10]:
df['Sex'].value_counts()

Sex
male      577
female    314
Name: count, dtype: Int64